# Thermalens - EDA and Modeling

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
from src.models import HeatRiskModel
import shap
import warnings
warnings.filterwarnings('ignore')

## 1. Load Synthetic Data

In [ ]:
try:
    gdf = gpd.read_file('../data/synthetic_features.geojson')
    print("Loaded data with shape:", gdf.shape)
    display(gdf.head())
except Exception as e:
    print("Data not found. Please run src/data.py first to generate it.")
    # Generate dummy data for illustration
    gdf = pd.DataFrame({
        'temperature_c': np.random.uniform(30, 45, 100),
        'humidity_pct': np.random.uniform(40, 90, 100),
        'ndvi': np.random.uniform(-0.1, 0.8, 100),
        'ndbi': np.random.uniform(-0.2, 0.7, 100),
        'vulnerability_index': np.random.uniform(0, 100, 100)
    })

## 2. Train Model and SHAP Explanations

In [ ]:
features = ['temperature_c', 'humidity_pct', 'ndvi', 'ndbi', 'vulnerability_index']
X = gdf[features].copy()
# Target simulating risk score
y = X['temperature_c']*0.5 - X['ndvi']*10 + X['ndbi']*10 + np.random.normal(0, 2, len(X))

model = HeatRiskModel('xgboost')
model.train(X, y)
print("Model trained.")

shap_values = model.explain(X)
shap.summary_plot(shap_values, X)